In [1]:
import numpy as np
import json
from torch.utils.data.dataset import TensorDataset
import  torch

In [3]:
with open("../activations/imagenet_train_hf/config.json") as f:
    config = json.load(f)


test = np.memmap(
    "../activations/imagenet_train_hf/embeddings.npy",
    dtype=np.float16,
    mode="r",
    shape=(config["dataset_size"], config["embedding_size"]),
)
labels = np.memmap(
    "../activations/imagenet_train_hf/labels.npy",
    dtype=np.int64,
    mode="r",
    shape=(config["dataset_size"],),
)


dataset = TensorDataset(torch.from_numpy(test), torch.from_numpy(labels))


/tmp/ipykernel_241503/4219942663.py:19: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  dataset = TensorDataset(torch.from_numpy(test), torch.from_numpy(labels))


In [8]:
from torch.utils.data.dataloader import DataLoader

dl = DataLoader(dataset, batch_size=8)

for batch in dl:
    tensors, labels = batch
    print(tensors.norm(p=2, dim=1))

    print(tensors.shape)
    print(labels.shape)
    break

tensor([10.8125,  8.8438,  9.7500, 10.8047, 10.8203, 10.5078, 11.1562, 10.2656],
       dtype=torch.float16)
torch.Size([8, 512])
torch.Size([8])


In [23]:
from ca_sae.sae.batch_top_k import BatchTopKSAE

sae = BatchTopKSAE.from_pretrained(
    "../checkpoints/test/BatchTopKSAEFirst/ae.pt", k=64, device="cuda"
)

In [32]:
for batch in dl:
    x = batch[0].to("cuda")
    f = sae.encode(x)

    x_hat = sae.decode(f)

    loss = (x - x_hat)**2
    # loss = torch.nn.functional.mse_loss(x_hat, x)

    print(loss)
    break

tensor([[1.1086e-03, 5.2704e-03, 9.0683e-03,  ..., 3.9293e-03, 2.0399e-03,
         2.6667e-02],
        [8.2772e-04, 8.4545e-02, 2.2840e-02,  ..., 1.0566e-03, 2.6218e-04,
         7.7541e-05],
        [6.1778e-03, 1.4070e-02, 1.3940e-03,  ..., 1.0446e-03, 9.9758e-03,
         3.1786e-03],
        ...,
        [1.4884e-03, 9.6716e-03, 5.0570e-03,  ..., 5.5749e-04, 2.7411e-03,
         3.1027e-03],
        [4.0905e-02, 3.9162e-02, 4.5243e-02,  ..., 6.4301e-02, 3.0533e-02,
         7.3943e-03],
        [2.1302e-02, 2.1648e-02, 1.1654e-02,  ..., 2.8811e-02, 8.7137e-03,
         1.2025e-02]], device='cuda:0', grad_fn=<PowBackward0>)


In [4]:
from lapsum.topk import soft_topk
import torch

x = torch.rand(size=(1, 4096)) * 10
x.requires_grad_(True)
k = torch.tensor([64.0]).unsqueeze(0)
k.requires_grad_(True)
alpha = torch.tensor([0.05])
alpha.requires_grad_(True)
selection = soft_topk(x, k, alpha)

for x in selection[0]:
    # if x > 1e-4:
    print(x.item())

1.2114076323283766e-14
0.0
0.0
0.0
3.7521853146644344e-34
3.040817667584853e-42
0.0
0.0
0.0
3.5827295167582283e-31
0.0
2.257334334103238e-15
0.0
4.615850211370962e-18
0.0
0.0
0.0
0.0
1.961817850054744e-44
0.0
0.0
0.0
2.6624670822171524e-44
0.0
0.0
0.004259106703102589
4.094676205782157e-26
0.0
1.7418748029740527e-05
1.263056559287351e-37
1.0121354833334997e-19
8.876595529727638e-05
0.0
0.0
0.0
1.5943049098575484e-28
0.00037881836760789156
1.1001098272358508e-26
0.0
0.00021510114311240613
0.0
0.0
0.0
0.0
7.906812099450116e-30
0.0
2.0739217272007293e-43
0.0
0.0
1.7909771464781195e-39
4.8168929963415676e-33
2.7797635822539473e-34
1.8183226452303397e-37
0.7978907823562622
9.606200561898292e-28
0.0
1.0413762734085008e-11
1.69124249443341e-19
3.8941396934266555e-30
4.616749482710958e-10
3.8842003391437174e-07
4.7105793420933965e-15
0.0
2.324520726917556e-19
1.10181855080111e-27
0.0
7.13300233674449e-29
5.642694882155384e-35
0.0
0.0006306180730462074
7.268815394145691e-41
2.7029696189617702e-

In [1]:
from ca_sae.sae.softsae import SoftSAE

sae  = SoftSAE(512, 4096, k=64, alpha=0.8)

sae.encode(torch.rand(size=(4, 512)), use_hard_topk=False)

NameError: name 'torch' is not defined

In [11]:
import  torch

linear = torch.nn.Sequential(torch.nn.Linear(10, 2))

linear[0].weight

Parameter containing:
tensor([[-0.0940,  0.1369, -0.2571, -0.0204, -0.2120, -0.2322,  0.1091,  0.0094,
          0.2384,  0.0445],
        [-0.1682,  0.1423,  0.1657, -0.2282,  0.1607, -0.3157, -0.0632,  0.1615,
         -0.2702, -0.0080]], requires_grad=True)

In [2]:
from datasets import load_dataset

In [4]:
from labels import IMAGENET2012_CLASSES

list(IMAGENET2012_CLASSES.values())

['tench, Tinca tinca',
 'goldfish, Carassius auratus',
 'great white shark, white shark, man-eater, man-eating shark, Carcharodon carcharias',
 'tiger shark, Galeocerdo cuvieri',
 'hammerhead, hammerhead shark',
 'electric ray, crampfish, numbfish, torpedo',
 'stingray',
 'cock',
 'hen',
 'ostrich, Struthio camelus',
 'brambling, Fringilla montifringilla',
 'goldfinch, Carduelis carduelis',
 'house finch, linnet, Carpodacus mexicanus',
 'junco, snowbird',
 'indigo bunting, indigo finch, indigo bird, Passerina cyanea',
 'robin, American robin, Turdus migratorius',
 'bulbul',
 'jay',
 'magpie',
 'chickadee',
 'water ouzel, dipper',
 'kite',
 'bald eagle, American eagle, Haliaeetus leucocephalus',
 'vulture',
 'great grey owl, great gray owl, Strix nebulosa',
 'European fire salamander, Salamandra salamandra',
 'common newt, Triturus vulgaris',
 'eft',
 'spotted salamander, Ambystoma maculatum',
 'axolotl, mud puppy, Ambystoma mexicanum',
 'bullfrog, Rana catesbeiana',
 'tree frog, tree-f

In [14]:
from ca_sae.sae.ca_sae import ClassAlignedSAE
from lapsum.topk import soft_topk
from ca_sae.sae.core import topk_per_row
import torch

sae = ClassAlignedSAE.from_pretrained("../checkpoints/test/ca_sae_v2/")
ks = torch.softmax(sae.budget_vector, dim=0) * 4096 * 5

m1 = soft_topk(sae.class_matrix, ks.unsqueeze(1), 0.01)

m2 = topk_per_row(sae.class_matrix, ks)

In [32]:
import torch

train_A  = torch.load("../experiment_results/empirical_matrices/batch_topk_63_train")

# for x in train_A[0]:
#     # print(x.item())

train_A.sum(dim=1).min()
m1.sum(dim=1).min()

tensor(0.1906, device='cuda:0', grad_fn=<MinBackward1>)